# 2) Adverbs: Do Great Writers Avoid Them?

**Goal:** Estimate -ly adverb rate and compare across the two texts.

# Setup: Load Texts

This notebook needs **Alice in Wonderland** and **Through the Looking-Glass** as input texts.

**How to provide the texts:**
1. Download books from Project Gutenberg (IDs 11 and 12) as txts. [go to https://www.gutenberg.org/ebooks/11 and https://www.gutenberg.org/ebooks/12]

2. Place two text files in the "data" folder with names:
   - `Wondeland.txt`  (Alice's Adventures in Wonderland)
   - `Looking-Glass.txt` (Through the Looking-Glass)

In [1]:
import re
from pathlib import Path

In [3]:
from pathlib import Path

def load_texts(local_crime: str = '../data/Crime-punishment.txt',
               local_brothers: str = '../data/The-BrothersKaramazov.txt'):
    """Load texts for 'Crime and Punishment' and 'The Brothers Karamazov'.

    Parameters
    ----------
    local_crime : str
        Path to 'Crime and Punishment' text file.
        Defaults to '../data/Crime-punishment.txt'.
    local_brothers : str
        Path to 'The Brothers Karamazov' text file.
        Defaults to '../data/The-BrothersKaramazov.txt'.

    Returns
    -------
    tuple[str, str]
        (crime_text, brothers_text).

    Raises
    ------
    FileNotFoundError
        If either file is missing.

    Extra Notes
    -----------
    - Uses UTF-8 encoding with `errors='ignore'` to avoid codec issues.
    """
    p1, p2 = Path(local_crime), Path(local_brothers)

    # Fail fast if file missing
    if not p1.exists():
        raise FileNotFoundError(
            f"Missing file: {p1}\n"
            "→ Please place 'Crime-punishment.txt' at this path or update load_texts(...)."
        )
    if not p2.exists():
        raise FileNotFoundError(
            f"Missing file: {p2}\n"
            "→ Please place 'The-BrothersKaramazov.txt' at this path or update load_texts(...)."
        )

    # Read both files safely
    crime_text = p1.read_text(encoding='utf-8', errors='ignore')
    brothers_text = p2.read_text(encoding='utf-8', errors='ignore')
    return crime_text, brothers_text


def normalize(text: str) -> str:
    """Normalize raw book text for tokenization.

    Steps
    -----
    1) Strip Project Gutenberg headers/footers if present
       (*** START ... *** END markers).
    2) Normalize newlines to '\n'.

    Parameters
    ----------
    text : str
        Raw text as loaded from disk (can be empty).

    Returns
    -------
    str
        Cleaned text suitable for tokenization and counting.
    """
    if not text:
        return ''
    # Clip to the main body if markers exist
    start = text.find('*** START')
    end = text.find('*** END')
    if start != -1 and end != -1 and end > start:
        text = text[start:end]
    return text.replace('\r\n', '\n')


# Load and normalize both novels
crime_raw, brothers_raw = load_texts()

crime = normalize(crime_raw)
brothers = normalize(brothers_raw)

print(f"'Crime and Punishment' chars: {len(crime):,} | 'The Brothers Karamazov' chars: {len(brothers):,}")


'Crime and Punishment' chars: 1,224,432 | 'The Brothers Karamazov' chars: 1,956,247


### Helpers: Tokenization

In [4]:
import re

WORD_RE = re.compile(r"[A-Za-z']+")  # keep apostrophes in words (e.g., don't -> don't)

def words(text: str):
    """Simple word tokenizer (lowercased, ASCII letters + apostrophes).

    Pros
    ----
    - Very fast and dependency-free.
    - Good enough for frequency/keyness demonstrations.

    Cons
    ----
    - No punctuation words, no sentence boundaries, no POS tags.
    - May treat possessives inconsistently across sources.

    Returns
    -------
    list[str]
        Lowercased word tokens.
    """
    return WORD_RE.findall(text.lower())


def sentences(text: str):
    """Naive sentence splitter using punctuation boundaries.

    Uses a regex to split on '.', '!', '?' followed by whitespace.
    Because this is heuristic, treat results as approximate.

    Returns
    -------
    list[str]
        Sentence-like strings.
    """
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


# Tokenize both novels
crime_words = words(crime)
brothers_words = words(brothers)

crime_sentences = sentences(crime)
brothers_sentences = sentences(brothers)

print(f"'Crime and Punishment' words: {len(crime_words):,} | 'The Brothers Karamazov' words: {len(brothers_words):,}")
print(f"'Crime and Punishment' sentences: {len(crime_sentences):,} | 'The Brothers Karamazov' sentences: {len(brothers_sentences):,}")


'Crime and Punishment' words: 214,498 | 'The Brothers Karamazov' words: 359,146
'Crime and Punishment' sentences: 16,994 | 'The Brothers Karamazov' sentences: 19,234


### Estimate -ly Adverb Rate

In [5]:
def adverb_rate(words):
    adverbs = [w for w in words if w.endswith('ly') and len(w) > 2]
    return len(adverbs), len(words), (len(adverbs) / len(words)) * 100


crime_adv, crime_total, crime_pct = adverb_rate(crime_words)
brothers_adv, brothers_total, brothers_pct = adverb_rate(brothers_words)

print(f"'Crime and Punishment': {crime_adv}/{crime_total} = {crime_pct:.2f}%")
print(f"'The Brothers Karamazov': {brothers_adv}/{brothers_total} = {brothers_pct:.2f}%")


'Crime and Punishment': 4092/214498 = 1.91%
'The Brothers Karamazov': 6767/359146 = 1.88%


**Prompt:** Inspect a sample of detected -ly words. Which are true adverbs vs. adjectives/nouns? How would you refine the rule?

In [6]:
# Show some adverbs from Crime and Punishment
adverbs_crime = [w for w in crime_words if w.endswith('ly') and len(w) > 2]
print(f"'Crime and Punishment' adverbs (first 20): {adverbs_crime[:20]}")
# Show some adverbs from The Brothers Karamazov
adverbs_brothers = [w for w in brothers_words if w.endswith('ly') and len(w) > 2]
print(f"'The Brothers Karamazov' adverbs (first 20): {adverbs_brothers[:20]}")


'Crime and Punishment' adverbs (first 20): ['cisely', 'beautifully', 'simply', 'ironically', 'completely', 'really', 'beautifully', 'family', 'family', 'brutally', 'similarly', 'closely', 'virtually', 'increasingly', 'simultaneously', 'psychologically', 'mysteriously', 'only', 'family', 'finally']
'The Brothers Karamazov' adverbs (first 20): ['carefully', 'verily', 'verily', 'family', 'only', 'hardly', 'frequently', 'worldly', 'apparently', 'fairly', 'family', 'easily', 'entirely', 'likely', 'probably', 'similarly', 'family', 'greatly', 'specially', 'passionately']


Let's use a smarter way to find adverbs:

In [9]:
# Run this cell once
import spacy
from spacy.cli import download

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    download("en_core_web_sm")          # downloads the small English model
    nlp = spacy.load("en_core_web_sm")  # try again


In [8]:
def strip_gutenberg_markup(t: str) -> str:
    t = re.sub(r"\b_+\b", " ", t)
    t = re.sub(r"_([A-Za-z]+)_", r"\1", t)
    return t

# Allow long texts
nlp.max_length = 2_500_000

# Process and show the first 20 adverbs from each book
for title, text in [
    ("Crime and Punishment", crime),
    ("The Brothers Karamazov", brothers)
]:
    text_clean = strip_gutenberg_markup(text)
    doc = nlp(text_clean)
    true_adverbs = [t.text for t in doc if t.pos_ == "ADV"]
    print(f"\n{title} — First 20 adverbs:\n{true_adverbs[:20]}\n")



Crime and Punishment — First 20 adverbs:
['still', 'as', 'well', 'cisely', 'as', 'well', 'beautifully', 'Simply', 'well', 'Ironically', 'so', 'much', 'thus', 'completely', 'really', 'beautifully', 'brutally', 'just', 'already', 'Similarly']


The Brothers Karamazov — First 20 adverbs:
['is', 'still', 'carefully', 'Verily', 'verily', 'alone', 'well', 'still', 'ago', 'only', 'so', 'hardly', 'yet', 'pretty', 'frequently', 'very', 'well', 'apparently', 'else', 'most']

